# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
- Chart chosen: Line chart.
- Why: It depicts the changes that occur in population over time in all continents.
- Design features: Africa is emphasized with golden orange colour and a thick line since it is the continent with the fastest growing population. The other continents are shown using lighter colors in order to keep the diagram simple. Population index is based on 1970 (100).

In [20]:
import pandas as pd
import plotly.express as px

# Population columns
population_columns = [
    '1970 Population',
    '1980 Population',
    '1990 Population',
    '2000 Population',
    '2010 Population',
    '2015 Population',
    '2020 Population',
    '2022 Population'
]

# Calculate total population for each continent
df_continent = df.groupby('Continent')[population_columns].sum().reset_index()

# Convert wide data into long format
df_long = df_continent.melt(
    id_vars='Continent',
    var_name='Year',
    value_name='Population'
)

# Extract year as a number
df_long['Year'] = df_long['Year'].str.extract(r'(\d{4})').astype(int)

# Calculate population index (1970 = 100)
df_index = df_long.copy()

base_population = (
    df_index[df_index['Year'] == 1970]
    .set_index('Continent')['Population']
)

df_index['Population Index'] = df_index.apply(
    lambda row: row['Population'] / base_population[row['Continent']] * 100,
    axis=1
)

# Colour palette: Africa highlighted
colors = {
    'Africa': '#E69F00',
    'Asia': '#83A9BC',
    'Europe': '#B39DDB',
    'North America': '#A5BFA0',
    'South America': '#D6A5B8',
    'Oceania': '#B8B0A0'
}

# Create line chart
fig = px.line(
    df_index,
    x='Year',
    y='Population Index',
    color='Continent',
    markers=True,
    color_discrete_map=colors,
    title='Population Growth by Continent: Africa Grew the Fastest'
)

# Highlight Africa with a thicker line
for trace in fig.data:
    if trace.name == 'Africa':
        trace.update(line=dict(width=4), marker=dict(size=9))
    else:
        trace.update(line=dict(width=2), marker=dict(size=5))

fig.update_layout(
    template='plotly_white',
    xaxis_title='Year',
    yaxis_title='Population Index (1970 = 100)',
    legend_title='Continent',
    title_x=0.5
)

fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE* Chart chosen: Horizontal bar chart.

Why: This makes it very convenient to compare and rank the 15 countries with the largest increase in population.

Design features: Countries are arranged from highest to lowest in population increase. Colors differentiate between the different countries, and the horizontal orientation helps in reading long country names.

In [10]:

import pandas as pd
import plotly.express as px

# Step 1: Calculate population increase
q2_df = df.copy()

q2_df['People Added'] = (
    q2_df['2022 Population'] - q2_df['1970 Population']
)

# Step 2: Select the top 15 countries
top15 = (
    q2_df.sort_values('People Added', ascending=False)
    .head(15)
    .copy()
)

# Step 3: Count African countries
africa_count = (top15['Continent'] == 'Africa').sum()

print("Number of African countries in the top 15:", africa_count)

# Step 4: Sort countries for the horizontal bar chart
top15 = top15.sort_values('People Added', ascending=True)

# Step 5: Soothing continent colour palette
continent_colours = {
    'Africa': '#9B91B8',
    'Asia': '#83A9BC',
    'Europe': '#D69B50',
    'North America': '#A5A0C4',
    'South America': '#78A69C',
    'Oceania': '#B78FA8'
}

# Step 6: Create the horizontal bar chart
fig = px.bar(
    data_frame=top15,
    x='People Added',
    y='Country',
    color='Continent',
    orientation='h',
    color_discrete_map=continent_colours,
    text='People Added',
    hover_data={
        'Continent': True,
        '1970 Population': ':,',
        '2022 Population': ':,',
        'People Added': ':,'
    },
    labels={
        'People Added': 'Additional People (1970–2022)',
        'Country': '',
        'Continent': 'Continent'
    },
    template='plotly_white'
)

# Step 7: Add labels and improve bar appearance
fig.update_traces(
    texttemplate='%{text:,.0f}',
    textposition='outside',
    cliponaxis=False,
    marker_line_width=0
)

# Step 8: Improve layout
fig.update_layout(
    title=dict(
        text=(
            f'The 15 Countries with the Largest Population Gains: '
            f'{africa_count} Are in Africa'
        ),
        x=0.5,
        xanchor='center',
        font=dict(size=19)
    ),
    height=650,
    font=dict(family='Arial', size=12),
    showlegend=True,
    legend=dict(
        title='Continent',
        orientation='h',
        y=-0.15,
        x=0.5,
        xanchor='center'
    ),
    margin=dict(l=110, r=100, t=90, b=100)
)

fig.update_xaxes(
    showgrid=False,
    title='Additional People (1970–2022)',
    tickformat=','
)

fig.update_yaxes(
    showgrid=False,
    title=''
)

fig.show()

# Step 9: Display the final ranking
display(
    top15[['Country', 'Continent', 'People Added']]
    .sort_values('People Added', ascending=False)
    .reset_index(drop=True)
)


Number of African countries in the top 15: 4


,Country,Continent,People Added
0,India,Asia,859671872
1,China,Asia,603352887
2,Pakistan,Asia,176533990
3,Nigeria,Africa,162971948
4,Indonesia,Asia,160272945
5,United States,North America,137961517
6,Brazil,South America,118943623
7,Bangladesh,Asia,103644512
8,Ethiopia,Africa,95071678
9,DR Congo,Africa,78858479


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
Chart chosen: Bubble scatter plot.



Why: It shows the relationship between land area and population in 2022.



Design choices: Each bubble represents a country, colours distinguish continents, and bubble size represents population density. Logarithmic axes help display countries with very different sizes and populations.

In [12]:

import pandas as pd
import numpy as np
import plotly.express as px

# Step 1: Prepare the data
q3_df = df[
    ['Country', 'Continent', 'Area',
     '2022 Population', 'Density']
].copy()

# Remove missing or invalid values
q3_df = q3_df.dropna()

q3_df = q3_df[
    (q3_df['Area'] > 0) &
    (q3_df['2022 Population'] > 0) &
    (q3_df['Density'] > 0)
]

# Step 2: Calculate logarithmic values
q3_df['Log Area'] = np.log10(q3_df['Area'])
q3_df['Log Population'] = np.log10(
    q3_df['2022 Population']
)

# Step 3: Calculate correlation
correlation = q3_df['Log Area'].corr(
    q3_df['Log Population']
)

print("Correlation between land area and population:")
print(round(correlation, 3))

# Step 4: Calculate overall trend
slope, intercept = np.polyfit(
    q3_df['Log Area'],
    q3_df['Log Population'],
    1
)

q3_df['Expected Log Population'] = (
    intercept + slope * q3_df['Log Area']
)

q3_df['Residual'] = (
    q3_df['Log Population'] -
    q3_df['Expected Log Population']
)

# Step 5: Identify unusual countries
above_trend = q3_df.nlargest(3, 'Residual')
below_trend = q3_df.nsmallest(3, 'Residual')

print("\nCountries above the overall trend:")
display(
    above_trend[
        ['Country', 'Continent', 'Area',
         '2022 Population', 'Density']
    ].reset_index(drop=True)
)

print("\nCountries below the overall trend:")
display(
    below_trend[
        ['Country', 'Continent', 'Area',
         '2022 Population', 'Density']
    ].reset_index(drop=True)
)

# Step 6: New pastel colour palette
continent_colours = {
    'Africa': '#B39DDB',          # Soft lavender
    'Asia': '#80CBC4',            # Muted turquoise
    'Europe': '#FFCC99',          # Soft peach
    'North America': '#90AFC5',   # Dusty blue
    'South America': '#A5BFA0',   # Sage green
    'Oceania': '#D6A5B8'           # Dusty pink
}

# Step 7: Create bubble scatter plot
fig = px.scatter(
    q3_df,
    x='Area',
    y='2022 Population',
    color='Continent',
    size='Density',
    size_max=28,
    hover_name='Country',
    color_discrete_map=continent_colours,
    hover_data={
        'Area': ':,',
        '2022 Population': ':,',
        'Density': ':.1f',
        'Continent': True
    },
    labels={
        'Area': 'Land Area (km²)',
        '2022 Population': 'Population in 2022',
        'Density': 'Population Density'
    },
    template='plotly_white'
)

# Step 8: Add overall trend line
x_line = np.linspace(
    q3_df['Log Area'].min(),
    q3_df['Log Area'].max(),
    100
)

y_line = intercept + slope * x_line

fig.add_scatter(
    x=10 ** x_line,
    y=10 ** y_line,
    mode='lines',
    name='Overall trend',
    line=dict(
        color='#555555',
        width=2,
        dash='dash'
    ),
    hoverinfo='skip'
)

# Step 9: Highlight unusual countries
outliers = pd.concat([
    above_trend,
    below_trend
])

fig.add_scatter(
    x=outliers['Area'],
    y=outliers['2022 Population'],
    mode='markers+text',
    text=outliers['Country'],
    textposition='top center',
    name='Unusual countries',
    marker=dict(
        symbol='diamond',
        size=11,
        color='#C47E91',
        line=dict(color='white', width=1)
    ),
    hovertemplate=(
        '<b>%{text}</b><br>'
        'Area: %{x:,.0f} km²<br>'
        'Population: %{y:,.0f}'
        '<extra></extra>'
    )
)

# Step 10: Customise layout
fig.update_layout(
    title=dict(
        text='Land Area and Population: A Continental Comparison',
        x=0.5,
        xanchor='center',
        font=dict(size=19)
    ),
    height=700,
    font=dict(family='Arial', size=12),
    hovermode='closest',
    legend=dict(
        title='Continent',
        orientation='h',
        y=-0.17,
        x=0.5,
        xanchor='center'
    ),
    margin=dict(l=80, r=40, t=90, b=120),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.update_xaxes(
    type='log',
    title='Land Area (km²) — Log Scale',
    showgrid=False
)

fig.update_yaxes(
    type='log',
    title='Population in 2022 — Log Scale',
    showgrid=True,
    gridcolor='#EEEEEE'
)

# Step 11: Display chart
fig.show()


Correlation between land area and population:
0.872

Countries above the overall trend:


,Country,Continent,Area,2022 Population,Density
0,Macau,Asia,30,695168,23172.2667
1,Singapore,Asia,710,5975689,8416.4634
2,Hong Kong,Asia,1104,7488865,6783.3922



Countries below the overall trend:


,Country,Continent,Area,2022 Population,Density
0,Greenland,North America,2166086,56466,0.0261
1,Falkland Islands,South America,12173,3780,0.3105
2,Niue,Oceania,260,1934,7.4385


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
Chart chosen: Violin plot.



Why: It shows how population density is distributed within each continent.



Design choices: Different colours distinguish continents. The inner box shows the median and spread, while individual points help identify differences between countries.

In [13]:

import pandas as pd
import plotly.express as px

# Step 1: Prepare the data
q4_df = df[
    ['Country', 'Continent', 'Density']
].copy()

# Remove missing and invalid values
q4_df = q4_df.dropna()

q4_df = q4_df[q4_df['Density'] > 0]

# Step 2: Calculate descriptive statistics
density_summary = (
    q4_df.groupby('Continent')['Density']
    .agg(['count', 'median', 'mean', 'min', 'max'])
    .round(2)
    .sort_values('median', ascending=False)
)

print("Population density summary by continent:")
display(density_summary)

# Step 3: Identify the continent with the highest median density
highest_median_continent = density_summary['median'].idxmax()

print(
    "\nContinent with the highest median population density:",
    highest_median_continent
)

# Step 4: Pastel colour palette
continent_colours = {
    'Africa': '#B39DDB',
    'Asia': '#80CBC4',
    'Europe': '#FFCC99',
    'North America': '#90AFC5',
    'South America': '#A5BFA0',
    'Oceania': '#D6A5B8'
}

# Step 5: Create violin plot
fig = px.violin(
    q4_df,
    x='Continent',
    y='Density',
    color='Continent',
    color_discrete_map=continent_colours,
    box=True,
    points='all',
    hover_name='Country',
    hover_data={
        'Density': ':.1f',
        'Continent': True
    },
    labels={
        'Continent': 'Continent',
        'Density': 'Population Density (people per km²)'
    },
    template='plotly_white'
)

# Step 6: Customise appearance
fig.update_traces(
    meanline_visible=True,
    marker=dict(
        size=4,
        opacity=0.55
    ),
    line=dict(width=1.5),
    opacity=0.8
)

# Step 7: Improve layout
fig.update_layout(
    title=dict(
        text='Population Density Distribution Across Continents',
        x=0.5,
        xanchor='center',
        font=dict(size=19)
    ),
    height=650,
    font=dict(family='Arial', size=12),
    showlegend=False,
    margin=dict(l=80, r=40, t=90, b=80),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.update_xaxes(
    title='Continent',
    showgrid=False
)

fig.update_yaxes(
    title='Population Density (people per km²)',
    showgrid=True,
    gridcolor='#EEEEEE'
)

fig.show()


Population density summary by continent:


,count,median,mean,min,max
Continent,,,,,
North America,40,221.42,272.76,0.03,1299.26
Asia,50,122.97,1025.02,2.17,23172.27
Europe,50,106.87,663.32,3.62,18234.50
Oceania,23,78.25,132.54,3.40,603.24
Africa,57,59.71,125.05,2.17,871.93
South America,14,17.79,20.97,0.31,65.02



Continent with the highest median population density: North America


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
Chart chosen: Choropleth map.



Why: It shows population growth and decline geographically between 2000 and 2022.



Design choices: Countries are coloured according to their percentage population change. A diverging colour scale helps distinguish growth from decline, while country borders make locations easier to identify.

In [16]:

import pandas as pd
import plotly.express as px

# Step 1: Prepare the data
q5_df = df[
    ['Country', 'CCA3', 'Continent',
     '2000 Population', '2022 Population']
].copy()

# Step 2: Calculate percentage population change
q5_df['Population Change (%)'] = (
    (q5_df['2022 Population'] - q5_df['2000 Population'])
    / q5_df['2000 Population']
) * 100

# Step 3: Classify population changes
q5_df['Change Category'] = q5_df[
    'Population Change (%)'
].apply(
    lambda x: 'Growth' if x > 0
    else ('Decline' if x < 0 else 'No Change')
)

# Step 4: Print summary
print("Population change summary (2000–2022):")

summary = (
    q5_df['Change Category']
    .value_counts()
    .reindex(['Growth', 'Decline', 'No Change'], fill_value=0)
)

display(summary.rename("Number of Countries").to_frame())

# Step 5: Display countries with population decline
declining = q5_df[
    q5_df['Population Change (%)'] < 0
].sort_values('Population Change (%)')

print("\nCountries with population decline:")

display(
    declining[
        ['Country', 'Continent', 'Population Change (%)']
    ].round(2).reset_index(drop=True)
)

# Step 6: Multi-colour pastel diverging palette
colour_scale = [
    [0.00, '#B76E79'],  # Dusty rose
    [0.15, '#C58AA0'],  # Muted pink
    [0.30, '#B39DDB'],  # Lavender
    [0.43, '#D7C6E8'],  # Pale purple
    [0.50, '#F5EBDD'],  # Warm cream
    [0.60, '#B9DCCF'],  # Mint
    [0.75, '#80CBC4'],  # Turquoise
    [0.88, '#83A9BC'],  # Dusty blue
    [1.00, '#557C91']   # Deep muted blue
]

# Step 7: Create choropleth map
fig = px.choropleth(
    q5_df,
    locations='CCA3',
    color='Population Change (%)',
    hover_name='Country',
    hover_data={
        'Continent': True,
        '2000 Population': ':,',
        '2022 Population': ':,',
        'Population Change (%)': ':.2f',
        'CCA3': False
    },
    color_continuous_scale=colour_scale,
    color_continuous_midpoint=0,
    labels={
        'Population Change (%)': 'Population Change (%)'
    },
    template='plotly_white'
)

# Step 8: Customise the map
fig.update_layout(
    title=dict(
        text='Population Change Across Countries (2000–2022)',
        x=0.5,
        xanchor='center',
        font=dict(size=20)
    ),
    height=680,
    font=dict(family='Arial', size=12),
    margin=dict(l=20, r=20, t=90, b=30),
    coloraxis_colorbar=dict(
        title='Population<br>Change (%)',
        ticksuffix='%',
        thickness=18,
        len=0.75
    ),
    geo=dict(
        showframe=False,
        showcoastlines=True,
        coastlinecolor='#B8B8B8',
        showland=True,
        landcolor='#F8F7F4',
        showcountries=True,
        countrycolor='white',
        showocean=True,
        oceancolor='#F1F5F7'
    )
)

fig.show()


Population change summary (2000–2022):


,Number of Countries
Change Category,
Growth,200
Decline,34
No Change,0



Countries with population decline:


,Country,Continent,Population Change (%)
0,Northern Mariana Islands,Oceania,-38.32
1,American Samoa,Oceania,-23.97
2,Lithuania,Europe,-23.60
3,Marshall Islands,Oceania,-23.34
4,Moldova,Europe,-23.02
5,Latvia,Europe,-22.65
6,Bosnia and Herzegovina,Europe,-22.63
7,Vatican City,Europe,-21.66
8,Wallis and Futuna,Oceania,-21.40
9,Ukraine,Europe,-18.78


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
Chart chosen: Treemap.



Why: It shows how the world's population is distributed across continents and individual countries in one chart.



Design choices: Rectangle size represents population. Countries are grouped by continent, and different colours distinguish the continents. Larger rectangles make the most populated countries easy to identify.

In [17]:

import pandas as pd
import plotly.express as px

# Step 1: Prepare the data
q6_df = df[
    ['Country', 'Continent', '2022 Population']
].copy()

q6_df = q6_df.dropna()

# Step 2: Calculate total population by continent
continent_totals = (
    q6_df.groupby('Continent')['2022 Population']
    .sum()
    .sort_values(ascending=False)
)

print("Population by continent in 2022:")
display(
    continent_totals.to_frame(
        name='2022 Population'
    ).style.format('{:,.0f}')
)

# Step 3: Find the largest continent
largest_continent = continent_totals.idxmax()

# Step 4: Calculate world population
world_population = q6_df['2022 Population'].sum()

print("\nTotal world population in 2022:",
      f"{world_population:,.0f}")

print("Largest continent by population:", largest_continent)

# Step 5: Create a colourful pastel palette
continent_colours = {
    'Africa': '#B39DDB',
    'Asia': '#80CBC4',
    'Europe': '#FFCC99',
    'North America': '#90AFC5',
    'South America': '#A5BFA0',
    'Oceania': '#D6A5B8'
}

# Step 6: Create treemap
fig = px.treemap(
    q6_df,
    path=[
        px.Constant('World'),
        'Continent',
        'Country'
    ],
    values='2022 Population',
    color='Continent',
    color_discrete_map=continent_colours,
    custom_data=['2022 Population'],
    template='plotly_white'
)

# Step 7: Customise labels and hover information
fig.update_traces(
    textinfo='label+percent parent',
    textfont=dict(
        family='Arial',
        size=13,
        color='#263238'
    ),
    marker=dict(
        line=dict(
            color='white',
            width=2
        )
    ),
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Population: %{value:,.0f}<br>'
        'Share of parent: %{percentParent:.1%}'
        '<extra></extra>'
    )
)

# Step 8: Improve layout
fig.update_layout(
    title=dict(
        text='Distribution of World Population by Continent and Country (2022)',
        x=0.5,
        xanchor='center',
        font=dict(size=19)
    ),
    height=700,
    font=dict(family='Arial', size=12),
    margin=dict(l=20, r=20, t=90, b=30),
    paper_bgcolor='white'
)

fig.show()


Population by continent in 2022:


,2022 Population
Continent,
Asia,"4,721,383,274"
Africa,"1,426,730,932"
Europe,"743,147,538"
North America,"600,296,136"
South America,"436,816,608"
Oceania,"45,038,554"



Total world population in 2022: 7,973,413,042
Largest continent by population: Asia


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*
Chart chosen: 100% stacked area chart.



Why: It shows how Africa, Asia, Europe, and the rest of the world changed their shares of the total world population over time.



Design choices: The vertical axis uses percentages from 0 to 100, so the total always represents the whole world population. Different colours distinguish the four groups and make changes in their shares easy to compare.

In [18]:

import pandas as pd
import plotly.express as px

# Step 1: Select population columns
population_columns = [
    '1970 Population',
    '1980 Population',
    '1990 Population',
    '2000 Population',
    '2010 Population',
    '2015 Population',
    '2020 Population',
    '2022 Population'
]

# Step 2: Calculate population totals by continent
continent_population = (
    df.groupby('Continent')[population_columns]
    .sum()
    .reset_index()
)

# Step 3: Convert data into long format
q7_df = pd.melt(
    continent_population,
    id_vars='Continent',
    value_vars=population_columns,
    var_name='Year',
    value_name='Population'
)

q7_df['Year'] = (
    q7_df['Year'].str.extract(r'(\d{4})').astype(int)
)

# Step 4: Group continents
main_continents = ['Africa', 'Asia', 'Europe']

q7_df['Region'] = q7_df['Continent'].apply(
    lambda x: x if x in main_continents else 'Rest of the world'
)

# Step 5: Calculate population by region and year
region_population = (
    q7_df.groupby(['Year', 'Region'])['Population']
    .sum()
    .reset_index()
)

# Step 6: Calculate total world population for each year
world_totals = (
    region_population.groupby('Year')['Population']
    .sum()
    .reset_index(name='World Population')
)

# Step 7: Calculate percentage share
region_population = region_population.merge(
    world_totals,
    on='Year'
)

region_population['World Share (%)'] = (
    region_population['Population']
    / region_population['World Population']
) * 100

# Step 8: Display population share table
share_table = region_population.pivot(
    index='Year',
    columns='Region',
    values='World Share (%)'
).round(2)

print("World population share by region (%):")
display(share_table)

# Step 9: Pastel colour palette
region_colours = {
    'Africa': '#B39DDB',
    'Asia': '#80CBC4',
    'Europe': '#FFCC99',
    'Rest of the world': '#90AFC5'
}

# Step 10: Create 100% stacked area chart
fig = px.area(
    region_population,
    x='Year',
    y='World Share (%)',
    color='Region',
    color_discrete_map=region_colours,
    category_orders={
        'Region': [
            'Africa',
            'Asia',
            'Europe',
            'Rest of the world'
        ]
    },
    labels={
        'Year': 'Year',
        'World Share (%)': 'Share of World Population (%)',
        'Region': 'Region'
    },
    template='plotly_white'
)

# Step 11: Customise the chart
fig.update_layout(
    title=dict(
        text='Changing Shares of World Population (1970–2022)',
        x=0.5,
        xanchor='center',
        font=dict(size=19)
    ),
    height=620,
    font=dict(family='Arial', size=12),
    hovermode='x unified',
    legend=dict(
        title='Region',
        orientation='h',
        y=-0.2,
        x=0.5,
        xanchor='center'
    ),
    margin=dict(l=80, r=40, t=90, b=110)
)

fig.update_xaxes(
    title='Year',
    showgrid=False,
    dtick=10
)

fig.update_yaxes(
    title='Share of World Population (%)',
    ticksuffix='%',
    range=[0, 100],
    showgrid=True,
    gridcolor='#EEEEEE'
)

fig.show()


World population share by region (%):


Region,Africa,Asia,Europe,Rest of the world
Year,,,,
1970,9.89,58.06,17.76,14.29
1980,10.84,59.32,15.59,14.25
1990,12.01,60.41,13.55,14.02
2000,13.32,60.76,11.81,14.10
2010,15.11,60.43,10.53,13.93
2015,16.18,60.05,9.99,13.79
2020,17.36,59.48,9.51,13.65
2022,17.89,59.21,9.32,13.57
